# Chapter 8.3: Adding Quantization Support for Your Model

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** the quantization method interface in vLLM
2. **Integrate** GPTQ quantization for weight loading and dequantization
3. **Integrate** AWQ quantization
4. **Support** FP8 quantization
5. **Make** your custom model work with quantized weights
6. **Test** quantization correctness
7. **Evaluate** performance impact of different quantization methods

---

## Prerequisites
- Chapter 8.1 (Model Registration)
- Chapter 8.2 (Implementing a New Model)
- Basic understanding of model quantization concepts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part8/chapter_8.3_quantization_support.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part8/chapter_8.3_quantization_support.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Overview: Why Quantization Matters

Large language models require significant GPU memory. Quantization reduces memory usage by storing weights in lower precision:

| Precision | Bits per Param | Memory for 7B Model | Relative Quality |
|-----------|---------------|---------------------|-------------------|
| FP32      | 32            | 28 GB               | Baseline          |
| FP16/BF16 | 16            | 14 GB               | ~Same as FP32     |
| FP8 (E4M3)| 8             | 7 GB                | Very close        |
| INT8      | 8             | 7 GB                | Good              |
| INT4 (GPTQ)| 4            | 3.5 GB              | Good with calibration |
| INT4 (AWQ) | 4            | 3.5 GB              | Good with calibration |
| INT3      | 3             | 2.6 GB              | Acceptable        |

```
Full Precision (FP16):              Quantized (INT4):
+---+---+---+---+---+---+           +--+--+--+--+--+--+
|16b|16b|16b|16b|16b|16b|   ->      |4b|4b|4b|4b|4b|4b| + scale + zero_point
+---+---+---+---+---+---+           +--+--+--+--+--+--+
    96 bits total                       24 bits + metadata
                                        ~4x compression
```

## 2. Quantization Architecture in vLLM

vLLM's quantization system is modular. Each quantization method implements the `QuantizationConfig` interface.

```
+----------------------------------+
|     QuantizationConfig (Base)    |
+----------------------------------+
| get_name() -> str                |
| get_supported_act_dtypes()       |
| get_min_capability() -> int      |
| get_quant_method() -> QuantMethod|
+----------------------------------+
         |
    +----+----+----+----+
    |    |    |    |    |
   GPTQ AWQ  FP8  BNB Marlin
```

In [ ]:
# The QuantizationConfig interface
# File: vllm/model_executor/layers/quantization/base_config.py

from abc import ABC, abstractmethod
from typing import Any, Dict, List, Optional, Type
import torch

class QuantizationConfig(ABC):
    """Base class for quantization configurations."""
    
    @abstractmethod
    def get_name(self) -> str:
        """Name of the quantization method (e.g., 'gptq', 'awq')."""
        ...
    
    @abstractmethod
    def get_supported_act_dtypes(self) -> List[torch.dtype]:
        """Supported activation data types."""
        ...
    
    @abstractmethod
    def get_min_capability(self) -> int:
        """Minimum GPU compute capability (e.g., 75 for Turing, 80 for Ampere)."""
        ...
    
    @classmethod
    @abstractmethod
    def from_config(cls, config: Dict[str, Any]) -> "QuantizationConfig":
        """Create from HuggingFace quantization config."""
        ...
    
    @abstractmethod
    def get_quant_method(self, layer, prefix):
        """Get the quantization method for a specific layer."""
        ...

print("QuantizationConfig abstract methods:")
for method_name in ['get_name', 'get_supported_act_dtypes', 'get_min_capability',
                     'from_config', 'get_quant_method']:
    print(f"  {method_name}()")

## 3. The QuantizeMethodBase Interface

Each quantization method provides a `QuantizeMethodBase` that handles the actual quantized operations.

In [ ]:
class QuantizeMethodBase(ABC):
    """
    Base class for quantization methods.
    
    Each method handles:
    1. Creating quantized parameters (weight, scales, zeros)
    2. Loading quantized weights from checkpoints
    3. Performing quantized matrix multiplication
    """
    
    @abstractmethod
    def create_weights(
        self,
        layer: torch.nn.Module,
        input_size: int,
        output_partition_sizes: List[int],
        input_size_per_partition: int,
        params_dtype: torch.dtype,
        **kwargs,
    ):
        """Create quantized weight parameters."""
        ...
    
    @abstractmethod
    def apply(
        self,
        layer: torch.nn.Module,
        x: torch.Tensor,
        bias: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Perform quantized matrix multiplication."""
        ...

print("QuantizeMethodBase interface:")
print("  create_weights(): Set up quantized parameters")
print("  apply(): Perform quantized matmul (replaces nn.Linear.forward)")

## 4. How Quantization Integrates with Linear Layers

When quantization is enabled, vLLM's linear layers use the quant method instead of regular matmul.

In [ ]:
import torch
import torch.nn as nn
from typing import Optional, List

class QuantizedLinear(nn.Module):
    """
    Simplified version of how vLLM's linear layers handle quantization.
    
    When quant_config is None: behaves like normal nn.Linear
    When quant_config is set: uses quantized weights + dequantization
    """
    
    def __init__(
        self,
        input_size: int,
        output_size: int,
        bias: bool = False,
        quant_config=None,
    ):
        super().__init__()
        self.input_size = input_size
        self.output_size = output_size
        self.quant_config = quant_config
        
        if quant_config is not None:
            # Quantized: create quantized weight parameters
            self.quant_method = quant_config.get_quant_method(self, "")
            self.quant_method.create_weights(
                self, input_size, [output_size], input_size,
                params_dtype=torch.float16
            )
        else:
            # Regular: standard float weight
            self.weight = nn.Parameter(
                torch.empty(output_size, input_size)
            )
        
        if bias:
            self.bias = nn.Parameter(torch.zeros(output_size))
        else:
            self.bias = None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.quant_config is not None:
            # Use quantized matmul
            return self.quant_method.apply(self, x, self.bias)
        else:
            # Regular matmul
            return nn.functional.linear(x, self.weight, self.bias)

# Demo without quantization
linear = QuantizedLinear(256, 512, bias=False, quant_config=None)
x = torch.randn(4, 256)
out = linear(x)
print(f"Regular linear: {x.shape} -> {out.shape}")
print(f"Weight shape: {linear.weight.shape}")
print(f"Weight dtype: {linear.weight.dtype}")
print(f"Memory: {linear.weight.numel() * 4 / 1024:.1f} KB (FP32)")

## 5. GPTQ Quantization

GPTQ (Generative Pre-trained Transformer Quantization) uses second-order information from calibration data to quantize weights to 4-bit or 3-bit.

In [ ]:
# GPTQ stores weights as packed integers with separate scales and zeros

class GPTQConfig:
    """
    GPTQ quantization configuration.
    
    File: vllm/model_executor/layers/quantization/gptq.py
    """
    
    def __init__(
        self,
        weight_bits: int = 4,        # Quantization bits (typically 4 or 3)
        group_size: int = 128,       # Group size for per-group quantization
        desc_act: bool = False,      # Whether to use activation ordering
        sym: bool = True,            # Symmetric quantization
    ):
        self.weight_bits = weight_bits
        self.group_size = group_size
        self.desc_act = desc_act
        self.sym = sym
        
        # How many weights are packed into one int32
        self.pack_factor = 32 // weight_bits  # e.g., 8 for 4-bit
    
    def get_name(self) -> str:
        return "gptq"

config = GPTQConfig(weight_bits=4, group_size=128)
print(f"GPTQ Configuration:")
print(f"  Bits: {config.weight_bits}")
print(f"  Group size: {config.group_size}")
print(f"  Pack factor: {config.pack_factor} (weights per int32)")
print(f"  Symmetric: {config.sym}")
print(f"\nCompression ratio: {16 / config.weight_bits}x vs FP16")

In [ ]:
import numpy as np

# Demonstrate GPTQ weight packing and dequantization

def simulate_gptq_quantize(
    weight: torch.Tensor,  # [out_features, in_features]
    bits: int = 4,
    group_size: int = 128,
):
    """
    Simulate GPTQ quantization.
    
    For each group of `group_size` input features:
    1. Compute scale and zero_point
    2. Quantize weights to `bits`-bit integers
    3. Pack multiple integers into int32
    """
    out_features, in_features = weight.shape
    num_groups = in_features // group_size
    pack_factor = 32 // bits
    
    # Reshape into groups
    weight_grouped = weight.reshape(out_features, num_groups, group_size)
    
    # Compute per-group scales and zeros
    w_min = weight_grouped.min(dim=-1).values  # [out, num_groups]
    w_max = weight_grouped.max(dim=-1).values
    
    max_int = 2**bits - 1
    scales = (w_max - w_min) / max_int  # [out, num_groups]
    zeros = -w_min / scales              # [out, num_groups]
    
    # Quantize
    quantized = torch.round(
        (weight_grouped - w_min.unsqueeze(-1)) / scales.unsqueeze(-1)
    ).clamp(0, max_int).to(torch.int32)
    
    # Pack into int32 (simplified)
    quantized_flat = quantized.reshape(out_features, in_features)
    packed_shape = (out_features, in_features // pack_factor)
    qweight = torch.zeros(packed_shape, dtype=torch.int32)
    
    for i in range(pack_factor):
        qweight |= quantized_flat[:, i::pack_factor] << (bits * i)
    
    return qweight, scales, zeros

def simulate_gptq_dequantize(
    qweight: torch.Tensor,  # [out, in/pack_factor] int32
    scales: torch.Tensor,    # [out, num_groups]
    zeros: torch.Tensor,     # [out, num_groups]
    bits: int = 4,
    group_size: int = 128,
):
    """Dequantize GPTQ weights."""
    pack_factor = 32 // bits
    out_features = qweight.shape[0]
    in_features = qweight.shape[1] * pack_factor
    max_int = 2**bits - 1
    
    # Unpack
    weight = torch.zeros(out_features, in_features, dtype=torch.float16)
    for i in range(pack_factor):
        weight[:, i::pack_factor] = (
            (qweight >> (bits * i)) & max_int
        ).to(torch.float16)
    
    # Dequantize
    num_groups = in_features // group_size
    weight = weight.reshape(out_features, num_groups, group_size)
    weight = (weight - zeros.unsqueeze(-1)) * scales.unsqueeze(-1)
    weight = weight.reshape(out_features, in_features)
    
    return weight

# Demo
original_weight = torch.randn(256, 256, dtype=torch.float16)
qweight, scales, zeros = simulate_gptq_quantize(original_weight, bits=4, group_size=128)
recovered_weight = simulate_gptq_dequantize(qweight, scales, zeros, bits=4, group_size=128)

error = (original_weight.float() - recovered_weight.float()).abs()
print(f"Original weight shape: {original_weight.shape}")
print(f"Packed weight shape:   {qweight.shape}")
print(f"Scales shape:          {scales.shape}")
print(f"\nMemory comparison:")
orig_bytes = original_weight.numel() * 2  # FP16
quant_bytes = qweight.numel() * 4 + scales.numel() * 2 + zeros.numel() * 2
print(f"  Original (FP16): {orig_bytes:,} bytes")
print(f"  Quantized (4-bit): {quant_bytes:,} bytes")
print(f"  Compression: {orig_bytes / quant_bytes:.2f}x")
print(f"\nQuantization error:")
print(f"  Mean absolute error: {error.mean():.6f}")
print(f"  Max absolute error:  {error.max():.6f}")

## 6. GPTQ Weight Loading in vLLM

In [ ]:
# How GPTQ weights are stored in HuggingFace checkpoints

GPTQ_WEIGHT_NAMES = '''
GPTQ Checkpoint Weight Names:
================================

For each quantized linear layer, there are 3-4 parameters:

1. qweight: Packed quantized weights
   Shape: [out_features, in_features // pack_factor]
   Dtype: int32
   Example: model.layers.0.self_attn.q_proj.qweight

2. scales: Per-group dequantization scales
   Shape: [num_groups, out_features] or [out_features, num_groups]
   Dtype: float16
   Example: model.layers.0.self_attn.q_proj.scales

3. qzeros: Packed zero points
   Shape: [num_groups, out_features // pack_factor]
   Dtype: int32
   Example: model.layers.0.self_attn.q_proj.qzeros

4. g_idx: Group index mapping (only if desc_act=True)
   Shape: [in_features]
   Dtype: int32
   Example: model.layers.0.self_attn.q_proj.g_idx

Standard linear layers (e.g., layernorm) remain in FP16.
'''

print(GPTQ_WEIGHT_NAMES)

In [ ]:
# How to handle GPTQ weights in load_weights()

def load_weights_with_gptq(
    model,
    weights,
    quant_config=None,
):
    """
    Load weights handling both FP16 and GPTQ quantized weights.
    
    GPTQ weights have special suffixes: .qweight, .scales, .qzeros, .g_idx
    """
    stacked_params_mapping = [
        ("qkv_proj", "q_proj", "q"),
        ("qkv_proj", "k_proj", "k"),
        ("qkv_proj", "v_proj", "v"),
        ("gate_up_proj", "gate_proj", 0),
        ("gate_up_proj", "up_proj", 1),
    ]
    
    params_dict = dict(model.named_parameters())
    loaded = set()
    
    for name, loaded_weight in weights:
        # Handle GPTQ-specific weight suffixes
        is_quantized_param = any(
            name.endswith(suffix)
            for suffix in [".qweight", ".scales", ".qzeros", ".g_idx"]
        )
        
        if is_quantized_param:
            # Extract the layer name and suffix
            # e.g., "model.layers.0.self_attn.q_proj.qweight"
            # -> layer_name = "model.layers.0.self_attn.q_proj"
            # -> suffix = "qweight"
            parts = name.rsplit(".", 1)
            layer_name = parts[0]
            suffix = parts[1]
            
            print(f"  Quantized param: {name}")
            print(f"    Layer: {layer_name}, Suffix: {suffix}")
            print(f"    Shape: {list(loaded_weight.shape)}, Dtype: {loaded_weight.dtype}")
            
            # Handle stacked params (e.g., q_proj -> qkv_proj)
            for vllm_name, hf_name, shard_id in stacked_params_mapping:
                if hf_name in layer_name:
                    merged_layer_name = layer_name.replace(hf_name, vllm_name)
                    param_name = f"{merged_layer_name}.{suffix}"
                    print(f"    Merged: {param_name} (shard={shard_id})")
                    break
            
            loaded.add(name)
        else:
            # Regular FP16 weight
            if name in params_dict:
                print(f"  FP16 param: {name}")
                loaded.add(name)
    
    return loaded

# Simulate loading GPTQ weights
print("Loading GPTQ quantized weights:")
print("=" * 60)

gptq_weights = [
    ("model.layers.0.self_attn.q_proj.qweight", torch.randint(0, 2**31, (256, 32), dtype=torch.int32)),
    ("model.layers.0.self_attn.q_proj.scales", torch.randn(2, 256, dtype=torch.float16)),
    ("model.layers.0.self_attn.q_proj.qzeros", torch.randint(0, 2**31, (2, 32), dtype=torch.int32)),
    ("model.layers.0.input_layernorm.weight", torch.randn(256, dtype=torch.float16)),
]

loaded = load_weights_with_gptq(None, gptq_weights)

## 7. AWQ Quantization

AWQ (Activation-Aware Weight Quantization) selectively preserves important weights at higher precision based on activation patterns.

In [ ]:
# AWQ key insight: Not all weights are equally important!
# Some weights have large corresponding activations and contribute more to output.
# AWQ scales these important weights UP before quantization, reducing their error.

def simulate_awq_quantize(
    weight: torch.Tensor,
    activation_scales: torch.Tensor,  # Per-channel activation magnitudes
    bits: int = 4,
    group_size: int = 128,
):
    """
    Simulate AWQ quantization.
    
    Key idea:
    1. Identify important channels (high activation magnitude)
    2. Scale important weights UP before quantization
    3. Scale activations DOWN correspondingly at runtime
    4. Net effect: important weights lose less precision
    """
    out_features, in_features = weight.shape
    
    # Step 1: Compute optimal scaling factors
    # Channels with large activations get larger scale factors
    scaling_factor = activation_scales.pow(0.5)  # Heuristic
    scaling_factor = scaling_factor / scaling_factor.max()  # Normalize
    scaling_factor = scaling_factor.clamp(min=0.01)  # Prevent zeros
    
    # Step 2: Scale weights (protect important channels)
    scaled_weight = weight * scaling_factor.unsqueeze(0)
    
    # Step 3: Standard quantization on scaled weights
    num_groups = in_features // group_size
    w_grouped = scaled_weight.reshape(out_features, num_groups, group_size)
    
    max_int = 2**bits - 1
    w_min = w_grouped.min(dim=-1).values
    w_max = w_grouped.max(dim=-1).values
    scales = (w_max - w_min) / max_int
    
    quantized = torch.round(
        (w_grouped - w_min.unsqueeze(-1)) / scales.unsqueeze(-1)
    ).clamp(0, max_int).to(torch.int32)
    
    return quantized, scales, scaling_factor

# Demo
weight = torch.randn(256, 256)
# Simulate activation scales (some channels have much larger activations)
act_scales = torch.abs(torch.randn(256))
act_scales[0:10] *= 100  # These channels are very important!

print("AWQ Quantization Demo")
print("=" * 60)
print(f"\nActivation scale statistics:")
print(f"  Min: {act_scales.min():.4f}")
print(f"  Max: {act_scales.max():.4f}")
print(f"  Mean: {act_scales.mean():.4f}")
print(f"  Important channels (>10x mean): {(act_scales > 10 * act_scales.mean()).sum()}")

qweight, scales, scaling = simulate_awq_quantize(weight, act_scales)
print(f"\nQuantized weight shape: {qweight.shape}")
print(f"Scaling factors range: [{scaling.min():.4f}, {scaling.max():.4f}]")

# Compare error for important vs unimportant channels
print(f"\nKey insight: Important weights get less quantization error")
print(f"because they are scaled UP before quantization.")

In [ ]:
# AWQ weight format in HuggingFace checkpoints

print("AWQ Checkpoint Weight Names:")
print("=" * 40)
print("""
AWQ uses a similar format to GPTQ:

1. qweight: Packed quantized weights
   model.layers.0.self_attn.q_proj.qweight [int32]

2. qzeros: Packed zero points  
   model.layers.0.self_attn.q_proj.qzeros [int32]

3. scales: Dequantization scales
   model.layers.0.self_attn.q_proj.scales [float16]

Difference from GPTQ:
- No g_idx (no activation reordering)
- Generally faster inference kernels (Marlin, GEMM)
- Better quality for same bit width (activation-aware)

In vLLM, AWQ uses the same Marlin kernel as GPTQ for
maximum performance.
""")

## 8. FP8 Quantization

FP8 is a newer quantization format supported on Hopper (H100) and Ada (L4/L40) GPUs. It uses 8-bit floating point instead of integer quantization.

In [ ]:
# FP8 quantization overview

print("FP8 Quantization")
print("=" * 60)
print("""
FP8 uses 8-bit floating point numbers:

E4M3 format (for weights):           E5M2 format (for activations):
  Sign: 1 bit                          Sign: 1 bit
  Exponent: 4 bits                     Exponent: 5 bits
  Mantissa: 3 bits                     Mantissa: 2 bits
  Range: [-448, 448]                   Range: [-57344, 57344]
  Precision: ~3 decimal digits         Precision: ~2 decimal digits

Advantages over INT4/INT8:
1. No calibration data needed (can quantize directly)
2. Hardware support on H100/L4/L40 GPUs
3. Near-FP16 quality with 2x speedup
4. Simpler implementation (just cast to fp8)

Memory savings:
  FP16: 2 bytes per weight
  FP8:  1 byte per weight + per-tensor scale
  Compression: ~2x
""")

# Simulate FP8 quantization
def simulate_fp8_quantize(weight: torch.Tensor):
    """Simulate FP8 quantization with per-tensor scaling."""
    # Compute per-tensor scale
    abs_max = weight.abs().max()
    fp8_max = 448.0  # E4M3 max value
    scale = fp8_max / abs_max
    
    # Scale and clamp to FP8 range
    scaled = weight * scale
    # In reality: torch.float8_e4m3fn
    quantized = scaled.clamp(-fp8_max, fp8_max)
    # Simulate reduced precision (round to ~3 mantissa bits)
    quantized = torch.round(quantized * 8) / 8
    
    return quantized, scale

def simulate_fp8_dequantize(quantized, scale):
    return quantized / scale

# Demo
weight = torch.randn(256, 256, dtype=torch.float16)
fp8_weight, scale = simulate_fp8_quantize(weight.float())
recovered = simulate_fp8_dequantize(fp8_weight, scale)

error = (weight.float() - recovered).abs()
print(f"\nFP8 Quantization Results:")
print(f"  Scale factor: {scale:.4f}")
print(f"  Mean absolute error: {error.mean():.6f}")
print(f"  Max absolute error: {error.max():.6f}")
print(f"  Memory: {weight.numel()} bytes (FP8) vs {weight.numel() * 2} bytes (FP16)")

## 9. Making Your Model Work with Quantization

In [ ]:
# The key to supporting quantization: pass quant_config to all linear layers

QUANT_MODEL_CODE = '''
class MyQuantizableModel(nn.Module):
    """
    A model that supports quantization.
    
    The ONLY change vs non-quantized: pass quant_config to linear layers.
    vLLM's linear layers handle the rest automatically!
    """
    
    def __init__(self, config, cache_config=None, quant_config=None):
        super().__init__()
        
        # IMPORTANT: Pass quant_config to all linear layers
        self.qkv_proj = QKVParallelLinear(
            config.hidden_size,
            config.hidden_size // config.num_attention_heads,
            config.num_attention_heads,
            config.num_key_value_heads,
            bias=False,
            quant_config=quant_config,  # <-- This is the key!
        )
        
        self.o_proj = RowParallelLinear(
            config.hidden_size,
            config.hidden_size,
            bias=False,
            quant_config=quant_config,  # <-- This too!
        )
        
        self.gate_up_proj = MergedColumnParallelLinear(
            config.hidden_size,
            [config.intermediate_size] * 2,
            bias=False,
            quant_config=quant_config,  # <-- And this!
        )
        
        self.down_proj = RowParallelLinear(
            config.intermediate_size,
            config.hidden_size,
            bias=False,
            quant_config=quant_config,  # <-- And this!
        )
        
        # NOTE: LayerNorm/RMSNorm do NOT get quant_config
        # They always stay in full precision
        self.layernorm = RMSNorm(config.hidden_size)
        
        # NOTE: Embeddings typically do NOT get quantized
        self.embed = VocabParallelEmbedding(config.vocab_size, config.hidden_size)

    # forward() is EXACTLY the same as non-quantized!
    # The quantized linear layers handle dequantization internally.
    def forward(self, input_ids, positions, kv_caches, attn_metadata):
        x = self.embed(input_ids)
        x = self.layernorm(x)
        qkv, _ = self.qkv_proj(x)  # Internally uses quantized matmul
        # ... etc
'''

print(QUANT_MODEL_CODE)
print("\nKey points:")
print("1. Pass quant_config to ALL linear layers")
print("2. Do NOT pass quant_config to norms and embeddings")
print("3. forward() code is IDENTICAL regardless of quantization")
print("4. Weight loading handles quantized params automatically")

## 10. Quantization-Aware Weight Loading

When loading quantized weights, the `load_weights()` method needs to handle the extra parameters (scales, zeros, etc.).

In [ ]:
QUANT_LOAD_WEIGHTS_CODE = '''
def load_weights(self, weights: Iterable[Tuple[str, torch.Tensor]]):
    """Load weights with quantization support."""
    
    stacked_params_mapping = [
        ("qkv_proj", "q_proj", "q"),
        ("qkv_proj", "k_proj", "k"),
        ("qkv_proj", "v_proj", "v"),
        ("gate_up_proj", "gate_proj", 0),
        ("gate_up_proj", "up_proj", 1),
    ]
    
    params_dict = dict(self.named_parameters())
    loaded = set()
    
    for name, loaded_weight in weights:
        # Skip RoPE frequencies
        if "rotary_emb.inv_freq" in name:
            continue
        
        # Handle GPTQ/AWQ suffixes
        # The weight loader handles .qweight, .scales, .qzeros automatically
        # through the param.weight_loader callback
        
        for (param_name, weight_name, shard_id) in stacked_params_mapping:
            if weight_name not in name:
                continue
            name = name.replace(weight_name, param_name)
            
            # Look up the parameter (might be qweight, scales, etc.)
            if name not in params_dict:
                break
            
            param = params_dict[name]
            # The weight_loader knows how to handle quantized weights
            weight_loader = param.weight_loader
            weight_loader(param, loaded_weight, shard_id)
            loaded.add(name)
            break
        else:
            if name in params_dict:
                param = params_dict[name]
                weight_loader = getattr(param, "weight_loader", default_weight_loader)
                weight_loader(param, loaded_weight)
                loaded.add(name)
    
    return loaded
'''

print(QUANT_LOAD_WEIGHTS_CODE)
print("\nImportant: The weight_loader callback on each parameter")
print("handles the complexity of loading quantized weights.")
print("You don't need to manually handle .qweight, .scales, etc.")

## 11. Testing Quantization Correctness

In [ ]:
# Quantization correctness testing framework

def test_quantization_correctness(
    original_weight: torch.Tensor,
    quantize_fn,
    dequantize_fn,
    method_name: str,
    tolerance: float = 0.1,
):
    """
    Test that quantization produces acceptable error.
    
    Tests:
    1. Reconstruction error (MSE, MAE)
    2. Output error (matmul with random input)
    3. Cosine similarity of outputs
    """
    print(f"\nTesting {method_name} Quantization")
    print("=" * 50)
    
    # Quantize and dequantize
    quantized = quantize_fn(original_weight)
    recovered = dequantize_fn(quantized)
    
    # Test 1: Weight reconstruction error
    mse = ((original_weight.float() - recovered.float()) ** 2).mean()
    mae = (original_weight.float() - recovered.float()).abs().mean()
    relative_error = mae / original_weight.float().abs().mean()
    
    print(f"\n1. Weight Reconstruction Error:")
    print(f"   MSE: {mse:.6f}")
    print(f"   MAE: {mae:.6f}")
    print(f"   Relative Error: {relative_error:.4%}")
    
    # Test 2: Output error
    input_tensor = torch.randn(32, original_weight.shape[1])
    original_output = input_tensor @ original_weight.float().t()
    quantized_output = input_tensor @ recovered.float().t()
    
    output_mse = ((original_output - quantized_output) ** 2).mean()
    output_mae = (original_output - quantized_output).abs().mean()
    
    print(f"\n2. Output Error (matmul with random input):")
    print(f"   MSE: {output_mse:.6f}")
    print(f"   MAE: {output_mae:.6f}")
    
    # Test 3: Cosine similarity
    cos_sim = torch.nn.functional.cosine_similarity(
        original_output.flatten().unsqueeze(0),
        quantized_output.flatten().unsqueeze(0)
    )
    print(f"\n3. Output Cosine Similarity: {cos_sim.item():.6f}")
    
    # Verdict
    passed = cos_sim.item() > (1 - tolerance)
    print(f"\nVerdict: {'PASS' if passed else 'FAIL'} (threshold: cosine > {1-tolerance})")
    return passed

# Test with our GPTQ simulation
weight = torch.randn(256, 256, dtype=torch.float16)

def my_quantize(w):
    return simulate_gptq_quantize(w, bits=4, group_size=128)

def my_dequantize(q):
    qweight, scales, zeros = q
    return simulate_gptq_dequantize(qweight, scales, zeros, bits=4, group_size=128)

test_quantization_correctness(
    weight, my_quantize, my_dequantize,
    "Simulated GPTQ (4-bit)", tolerance=0.01
)

## 12. Performance Comparison of Quantization Methods

In [ ]:
import time

# Performance comparison (simulated - real performance depends on GPU kernels)

def benchmark_matmul(weight: torch.Tensor, input_tensor: torch.Tensor, n_iters: int = 100):
    """Benchmark matrix multiplication."""
    # Warmup
    for _ in range(10):
        _ = input_tensor @ weight.t()
    
    start = time.time()
    for _ in range(n_iters):
        _ = input_tensor @ weight.t()
    elapsed = time.time() - start
    
    return elapsed / n_iters

# Create test tensors
hidden_size = 4096
batch_size = 32

print("Quantization Method Comparison")
print("=" * 70)
print(f"{'Method':<15} {'Bits':>5} {'Memory (MB)':>12} {'Quality':>10} {'Speed':>10}")
print("-" * 70)

methods = [
    ("FP32",   32, 1.0,   "Baseline", "1.0x"),
    ("FP16",   16, 0.5,   "~Same",    "1.5-2x"),
    ("BF16",   16, 0.5,   "~Same",    "1.5-2x"),
    ("FP8",     8, 0.25,  "Very good", "2-3x"),
    ("INT8",    8, 0.25,  "Good",      "1.5-2x"),
    ("GPTQ-4",  4, 0.125, "Good",      "2-4x"),
    ("AWQ-4",   4, 0.125, "Good+",     "2-4x"),
    ("GPTQ-3",  3, 0.094, "Acceptable", "2-3x"),
]

base_memory = hidden_size * hidden_size * 4 / (1024**2)  # FP32 memory in MB

for name, bits, mem_ratio, quality, speed in methods:
    memory = base_memory * mem_ratio
    print(f"{name:<15} {bits:>5} {memory:>10.1f}MB {quality:>10} {speed:>10}")

print(f"\nNote: Speed numbers are approximate and depend on:")
print(f"  - GPU architecture (H100 has native FP8 support)")
print(f"  - Batch size (larger batches benefit more from quantization)")
print(f"  - Kernel implementation (Marlin, CUTLASS, etc.)")

## 13. Marlin Kernel: High-Performance INT4 Matmul

vLLM uses the Marlin kernel for maximum INT4 performance.

In [ ]:
print("Marlin Kernel for INT4 Quantization")
print("=" * 60)
print("""
Marlin is a highly optimized CUDA kernel for INT4 x FP16 matrix multiplication.

Features:
1. Near-FP16 speed for INT4 quantized weights
2. Supports both GPTQ and AWQ formats
3. Works with Ampere (A100) and Hopper (H100) GPUs
4. Handles tensor parallelism

How it works:
- Keeps weights in packed INT4 format in GPU memory
- Dequantizes on-the-fly during matrix multiply
- Uses GPU tensor cores efficiently
- Achieves up to 4x memory reduction with <5% speed overhead

Usage in vLLM:
- Automatically selected for GPTQ/AWQ models on compatible GPUs
- Requires specific weight format (repacking may be needed)
- Compatible with TP (tensor parallelism)

Performance (A100, batch=1):
  FP16:  ~300 TFLOPS
  INT4 Marlin: ~280 TFLOPS effective (with 4x less memory bandwidth)
""")

## 14. Choosing the Right Quantization Method

In [ ]:
print("Quantization Method Selection Guide")
print("=" * 60)
print("""
Decision tree for choosing a quantization method:

1. Do you have an H100/L4/L40 GPU?
   YES -> Use FP8 (best quality/speed tradeoff, no calibration needed)
   NO  -> Continue to step 2

2. Do you need maximum quality?
   YES -> Use FP16/BF16 (no quantization)
   NO  -> Continue to step 3

3. Do you have a calibration dataset?
   YES -> Choose between GPTQ and AWQ:
          - AWQ: Generally better quality, faster quantization
          - GPTQ: More battle-tested, wider support
   NO  -> Use FP8 if available, or INT8 (SmoothQuant)

4. Do you need 3-bit quantization?
   YES -> Use GPTQ-3bit (but expect quality degradation)
   NO  -> Use 4-bit (GPTQ or AWQ)

Common configurations:
  Production (quality): FP8 on H100, AWQ-4bit on A100
  Production (speed):   AWQ-4bit with Marlin kernel
  Development/testing:  FP16/BF16
  Edge deployment:      GPTQ-4bit or GPTQ-3bit
""")

## 15. Source Code Walkthrough: How vLLM Detects Quantization

In [ ]:
# How vLLM detects and applies quantization

DETECTION_CODE = '''
# File: vllm/config.py (simplified)

class ModelConfig:
    def _get_quantization_config(self):
        """Detect quantization from model config."""
        
        # Method 1: Explicit --quantization flag
        if self.quantization:
            return self.quantization  # "gptq", "awq", "fp8", etc.
        
        # Method 2: Detect from HuggingFace config
        hf_config = self.hf_config
        
        # Check for GPTQ
        quant_config = getattr(hf_config, "quantization_config", None)
        if quant_config:
            quant_method = quant_config.get("quant_method", "")
            if quant_method == "gptq":
                return "gptq"
            elif quant_method == "awq":
                return "awq"
        
        # Check for BitsAndBytes
        if getattr(hf_config, "quantization_config", {}).get("load_in_4bit"):
            return "bitsandbytes"
        
        return None  # No quantization

# File: vllm/model_executor/model_loader/loader.py (simplified)

def _get_quant_config(model_config, model_class):
    """Create QuantizationConfig from model config."""
    quant_method = model_config.quantization
    
    if quant_method is None:
        return None
    
    # Map method name to config class
    QUANT_CONFIG_MAP = {
        "gptq": GPTQConfig,
        "awq": AWQConfig,
        "fp8": FP8Config,
        "marlin": MarlinConfig,
        "gptq_marlin": GPTQMarlinConfig,
        "awq_marlin": AWQMarlinConfig,
        "bitsandbytes": BitsAndBytesConfig,
        "squeezellm": SqueezeLLMConfig,
    }
    
    config_class = QUANT_CONFIG_MAP[quant_method]
    return config_class.from_config(model_config.hf_config.quantization_config)
'''

print(DETECTION_CODE)

## 16. Adding Quantization to Your Custom Model: Complete Example

In [ ]:
import torch
import torch.nn as nn

class QuantizationAwareModel(nn.Module):
    """
    Complete example of a model with quantization support.
    
    Key principle: The model code is IDENTICAL for quantized and non-quantized.
    Only the linear layer constructor changes behavior based on quant_config.
    """
    
    def __init__(self, config, cache_config=None, quant_config=None):
        super().__init__()
        self.config = config
        self.quant_config = quant_config
        
        hidden_size = config.hidden_size
        intermediate_size = config.intermediate_size
        num_heads = config.num_attention_heads
        num_kv_heads = getattr(config, 'num_key_value_heads', num_heads)
        head_dim = hidden_size // num_heads
        
        # Embedding (NOT quantized)
        self.embed = nn.Embedding(config.vocab_size, hidden_size)
        
        # Attention projections (quantized if quant_config is set)
        qkv_size = (num_heads + 2 * num_kv_heads) * head_dim
        self.qkv_proj = nn.Linear(hidden_size, qkv_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        
        # MLP projections (quantized if quant_config is set)
        self.gate_up_proj = nn.Linear(hidden_size, 2 * intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        
        # Norms (NOT quantized)
        self.input_norm = nn.LayerNorm(hidden_size)
        self.post_attn_norm = nn.LayerNorm(hidden_size)
        
        # LM head (NOT quantized)
        self.lm_head = nn.Linear(hidden_size, config.vocab_size, bias=False)
        
        if quant_config:
            print(f"Model created with {quant_config.get_name()} quantization")
        else:
            print("Model created without quantization")
    
    def forward(self, input_ids, positions, kv_caches, attn_metadata):
        # EXACTLY the same code for quantized and non-quantized!
        x = self.embed(input_ids)
        
        # Attention
        residual = x
        x = self.input_norm(x)
        qkv = self.qkv_proj(x)  # Uses quantized matmul if quant_config set
        # ... attention computation ...
        x = self.o_proj(x)  # Uses quantized matmul if quant_config set
        x = x + residual
        
        # MLP
        residual = x
        x = self.post_attn_norm(x)
        gate_up = self.gate_up_proj(x)  # Quantized matmul
        intermediate_size = gate_up.shape[-1] // 2
        gate = torch.sigmoid(gate_up[..., :intermediate_size])
        up = gate_up[..., intermediate_size:]
        x = self.down_proj(gate * up)  # Quantized matmul
        x = x + residual
        
        # Output
        logits = self.lm_head(x)
        return logits

# Test without quantization
class TestConfig:
    vocab_size = 1000
    hidden_size = 256
    intermediate_size = 512
    num_attention_heads = 4
    num_key_value_heads = 2

config = TestConfig()
model = QuantizationAwareModel(config, quant_config=None)

x = torch.randint(0, 1000, (4, 16))
positions = torch.arange(16).unsqueeze(0).expand(4, -1)
output = model(x, positions, [], None)
print(f"Output shape: {output.shape}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

## 17. Exercises

### Exercise 1: Implement Simple INT8 Quantization
Implement per-channel INT8 quantization and dequantization. Test the reconstruction error.

### Exercise 2: Add GPTQ Support to Custom Model
Given the `QuantizationAwareModel` above, write the `load_weights()` method that handles both regular FP16 and GPTQ quantized weights.

### Exercise 3: Quantization Quality Benchmark
Compare the output quality of different quantization methods on a simple model:
- No quantization (baseline)
- INT8 per-tensor
- INT8 per-channel
- INT4 per-group (group_size=128)
- INT4 per-group (group_size=32)

In [ ]:
# Exercise 1: Starter code for INT8 quantization

def int8_per_channel_quantize(weight: torch.Tensor):
    """
    Per-channel INT8 quantization.
    
    For each output channel:
    1. Compute scale = max(abs(weight[channel])) / 127
    2. Quantize: q = round(weight / scale)
    3. Clamp to [-128, 127]
    """
    # TODO: Implement
    # Hint: scale shape should be [out_features, 1]
    abs_max = weight.abs().max(dim=1, keepdim=True).values
    scale = abs_max / 127.0
    quantized = torch.round(weight / scale).clamp(-128, 127).to(torch.int8)
    return quantized, scale

def int8_dequantize(quantized: torch.Tensor, scale: torch.Tensor):
    """Dequantize INT8 weights."""
    return quantized.float() * scale

# Test
w = torch.randn(256, 256)
qw, s = int8_per_channel_quantize(w)
rw = int8_dequantize(qw, s)
error = (w - rw).abs().mean()
print(f"INT8 per-channel quantization error: {error:.6f}")
print(f"Compression: {w.numel() * 4} bytes -> {qw.numel() + s.numel() * 4} bytes")

## Summary

In this notebook, we covered:

1. **QuantizationConfig**: The interface all quantization methods implement
2. **GPTQ**: 4-bit post-training quantization with calibration
3. **AWQ**: Activation-aware 4-bit quantization
4. **FP8**: 8-bit floating point for modern GPUs
5. **Integration**: Pass `quant_config` to linear layers, keep forward() the same
6. **Weight Loading**: Handle .qweight, .scales, .qzeros parameters
7. **Testing**: Verify reconstruction error and output quality
8. **Performance**: Different methods trade off quality, speed, and memory

### Key Insight
Supporting quantization in your custom model is mostly about passing `quant_config` to the right layers. The linear layers handle everything else automatically.

### Next: Chapter 8.4 -- Multi-Modal Model Integration